# Project: Live Streaming Pipeline 

## Welcome!
- This notebook provides solutions for the weekly project about building a live streaming pipeline with Kafka and ksqlDB, using the Wikimedia RecentChanges stream.
- **Prerequisites**:
  - Producer running (sending Wikipedia data; stop/start via Jupyter kernel).
  - `WIKI_RAW_STREAM` created.
- **Setup**:
  - Ensure Docker services are running: `docker compose up -d`.
  - Install packages: `!pip install kafka-python` incase jupyter is needed in excercise.
  - All excercises to be done in ksqlDB - Check: `docker exec -it ksql-server ksql http://host.docker.internal:8088`.
  - Use `host.docker.internal` for macOS/Windows; for Linux, use your Kafka/ksqlDB server IP.
- **Note**: Outputs are indicative (live data varies). Adjust `LIMIT` as needed.


---

In [3]:
!pip install requests kafka-python

### Exercise 1: Filter Main Namespace Edits
#### What to Do
- Create a ksqlDB stream to filter `WIKI_RAW_STREAM` for edits in the main namespace (`wiki_namespace = 0`).
- Run in ksqlDB CLI:

In [ ]:
CREATE STREAM MAIN_NAMESPACE AS
  SELECT 
    title,           
    user,           
    length_new - length_old AS size_change,
    ROWTIME AS timestamp      
  FROM WIKI_RAW_STREAM
  WHERE wiki_namespace = 0
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, size_change
FROM MAIN_NAMESPACE
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+-------------------------+-------------------------+-------------------------+
|TITLE                    |USER                     |SIZE_CHANGE              |
+-------------------------+-------------------------+-------------------------+
|Q123760944               |Estopedist1              |427                      |
|Q136342720               |StructuraMuseion         |458                      |
|dune coon                |1.140.160.117            |26                       |
+-------------------------+-------------------------+-------------------------+
```

### Exercise 2: Filter New Page Creations
#### What to Do
- Create a ksqlDB stream to filter `WIKI_RAW_STREAM` for new page creations (`type = 'new'`).
- Run in ksqlDB CLI:

In [ ]:
CREATE STREAM NEW_PAGES AS
  SELECT 
    title,
    user,
    wiki,
    ROWTIME AS timestamp
  FROM WIKI_RAW_STREAM
  WHERE type = 'new'
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, wiki
FROM NEW_PAGES
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+------------------------+------------------+-------+
|TITLE                  |USER              |WIKI   |
+------------------------+------------------+-------+
|New AI algorithm       |TechCreator       |enwiki |
|Quantum theory X       |SciUser           |frwiki |
|Crypto startup         |NewEditor         |eswiki |
+------------------------+------------------+-------+
```

### Exercise 3: Filter Category Page Edits
#### What to Do
- Create a ksqlDB stream to filter `WIKI_RAW_STREAM` for category page edits (`wiki_namespace = 14`).
- Run in ksqlDB CLI:

In [ ]:
CREATE STREAM CATEGORY_EDITS AS
  SELECT 
    title,
    user,
    wiki,
    ROWTIME AS timestamp
  FROM WIKI_RAW_STREAM
  WHERE wiki_namespace = 14
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, wiki
FROM CATEGORY_EDITS
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+-------------------------+-------------------------+-------------------------+
|TITLE                    |USER                     |WIKI                     |
+-------------------------+-------------------------+-------------------------+
|Category:Taken with Penta|Rkieferbot               |commonswiki              |
|x K100D                  |                         |                         |
|Category:All articles wit|Tobyhoward               |enwiki                   |
|h unsourced statements   |                         |                         |
|Category:Articles with un|Tobyhoward               |enwiki                   |
|sourced statements from S|                         |                         |
|eptember 2025            |                         |                         |
+-------------------------+-------------------------+-------------------------+
```

### Exercise 4: Filter Edits with Size Change of 100 Bytes or Less
#### What to Do
- Create a ksqlDB stream to filter `WIKI_RAW_STREAM` for edits with an absolute size change of 100 bytes or less.
- Run in ksqlDB CLI:

In [ ]:
CREATE OR REPLACE STREAM MINOR_EDITS_LESS_THAN_100 AS
  SELECT 
    title, 
    user, 
    wiki,
    ABS(length_new - length_old) AS edit_length
  FROM WIKI_RAW_STREAM
  WHERE ABS(length_new - length_old) <= 100 AND ABS(length_new - length_old) > 0
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, wiki, edit_length
FROM MINOR_EDITS_LESS_THAN_100
EMIT CHANGES
LIMIT 6;

**Expected Output**:
```
+-----------------------------------------------------------+----------------+--------------+-------------+
| TITLE                                                     | USER           | WIKI         | EDIT_LENGTH |
+-----------------------------------------------------------+----------------+--------------+-------------+
| Szövet (biológia)                                         | Crimea         | huwiki       | 29          |
| File:Canyon de Chelly 7-27-09 (5).JPG                     | Rkieferbot     | commonswiki  | 36          |
| Category:Acco Festival of Alternative Israeli Theatre     | Gveret Tered   | commonswiki  | 30          |
| Ariana Grande                                             | Sailorsfriend  | dewiki       | 4           |
| File:Rotterdamsche courant 29-06-1855 (IA ddd 010396554   | OlafJanssen    | commonswiki  | 91           |
| Karl Peter Leffler                                        | Marcus.linneberg| svwiki      | 40           |
+-----------------------------------------------------------+----------------+--------------+-------------+
```

### Exercise 5: Filter Edits Without Comments in English Wikipedia
#### What to Do
- Create a ksqlDB stream to filter `WIKI_RAW_STREAM` for edits with no comments in the English Wikipedia (`enwiki`).
- Run in ksqlDB CLI:

In [ ]:
CREATE STREAM NO_COMMENT_EDITS AS
  SELECT 
    title,
    user,
    wiki,
    ROWTIME AS timestamp
  FROM WIKI_RAW_STREAM
  WHERE (comment IS NULL OR comment = '') AND wiki = 'enwiki'
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, wiki
FROM NO_COMMENT_EDITS
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+-----------------------------------------------+--------------------------------------+-----------+-----------------+
| TITLE                                         | USER                                 | WIKI      | TIMESTAMP       |
+-----------------------------------------------+--------------------------------------+-----------+-----------------+
| Template:Iranian submissions for the Academy  | Οἶδα                                 | enwiki    | 1758440822198   |
| Award for Best International Feature Film     |                                      |           |                 |
| Tom Platt                                     | 2A02:C7C:6E19:9600:875:D0DA:1DE1:6741| enwiki    | 1758440833207   |
| Christopher O'Connell                         | 82.32.49.206                         | enwiki    | 1758440833780   |
| Template:National sports teams of the         | Lancepark                            | enwiki    | 1758440835906   |
| Dominican Republic                            |                                      |           |                 |
| R. Madhavan                                   | 2409:408D:794:849C:0:0:D55:98B1      | enwiki    | 1758440835968   |
| Draft:Menghai Mosque                          |                                      |           |                 |
+-----------------------------------------------+--------------------------------------+-----------+-----------------+
```

### Exercise 6: Count Edits by Namespace
#### What to Do
- Create a ksqlDB table to count edits by `wiki_namespace` in 5-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE OR REPLACE TABLE NAMESPACE_COUNTS AS
  SELECT
    wiki_namespace,
    WINDOWSTART AS window_start,
    WINDOWEND AS window_end,
    COUNT(*) AS edit_count
  FROM WIKI_RAW_STREAM
  WINDOW TUMBLING (SIZE 5 MINUTES)
  GROUP BY wiki_namespace
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT wiki_namespace, edit_count
FROM NAMESPACE_COUNTS
EMIT CHANGES LIMIT 7;

**Expected Output**:
```
+---------------+---------------------+---------------------+-----------+
| WIKI_NAMESPACE| WINDOWSTART         | WINDOWEND           | EDIT_COUNT|
+---------------+---------------------+---------------------+-----------+
| 0             | 1758440700000       | 1758441000000       | 54        |
| 1             | 1758440700000       | 1758441000000       | 1         |
| 2             | 1758440700000       | 1758441000000       | 8         |
| 3             | 1758440700000       | 1758441000000       | 25        |
| 4             | 1758440700000       | 1758441000000       | 14        |
| 6             | 1758440700000       | 1758441000000       | 69        |
| 14            | 1758440700000       | 1758441000000       | 104       |
+---------------+---------------------+---------------------+-----------+
```

### Exercise 7: Count Edits by Server Name
#### What to Do
- Create a ksqlDB table to aggregate edit counts by `server_name` in 2-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE SERVER_ACTIVITY AS
  SELECT
    server_name,
    COUNT(*) AS total_edits
  FROM WIKI_RAW_STREAM
  WINDOW TUMBLING (SIZE 2 MINUTES)
  GROUP BY server_name
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT server_name, total_edits
FROM SERVER_ACTIVITY
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+--------------------+-------------+
| SERVER_NAME        | TOTAL_EDITS |
+--------------------+-------------+
| fr.wiktionary.org  |           1 |
| de.wikisource.org  |           2 |
| ceb.wikipedia.org  |           1 |
| ru.wikipedia.org   |           1 |
| de.wiktionary.org  |           2 |
+--------------------+-------------+
```

### Exercise 8: Count New Pages by Wiki
#### What to Do
- Create a ksqlDB table to count new page creations by `wiki` in 10-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE NEW_PAGE_COUNTS AS
  SELECT
    wiki,
    COUNT(*) AS new_page_count
  FROM NEW_PAGES
  WINDOW TUMBLING (SIZE 10 MINUTES)
  GROUP BY wiki
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT wiki, new_page_count
FROM NEW_PAGE_COUNTS
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+-------+----------------+
|WIKI   |NEW_PAGE_COUNT  |
+-------+----------------+
|enwiki |10              |
|eswiki |3               |
|frwiki |2               |
|dewiki |4               |
|jawiki |1               |
+-------+----------------+
```

### Exercise 9: Identify Frequent Editors
#### What to Do
- Create a ksqlDB table to find users with 3 or more edits in `MAIN_NAMESPACE` in 10-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE FREQUENT_EDITORS AS
  SELECT
    user,
    COUNT(*) AS edit_count
  FROM MAIN_NAMESPACE
  WINDOW TUMBLING (SIZE 10 MINUTES)
  GROUP BY user
  HAVING COUNT(*) >= 3
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT user, edit_count
FROM FREQUENT_EDITORS
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+------------------+------------+
| USER             | EDIT_COUNT |
+------------------+------------+
| Aka              | 3          |
| BorkedBot        | 6          |
| Instance of Bot  | 4          |
| Lsjbot           | 3          |
| Pmartinolli      | 28         |
+------------------+------------+
```

### Exercise 10: Analyze Edit Size Distribution
#### What to Do
- Create a ksqlDB table to categorize edits in `MAIN_NAMESPACE` by size change in 3-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE EDIT_SIZE_DISTRIBUTION AS
  SELECT
    CASE
      WHEN ABS(size_change) <= 100 THEN 'Small'
      WHEN ABS(size_change) <= 500 THEN 'Medium'
      ELSE 'Large'
    END AS size_category,
    COUNT(*) AS edit_count
  FROM MAIN_NAMESPACE
  WINDOW TUMBLING (SIZE 3 MINUTES)
  GROUP BY CASE
             WHEN ABS(size_change) <= 100 THEN 'Small'
             WHEN ABS(size_change) <= 500 THEN 'Medium'
             ELSE 'Large'
           END
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT size_category, edit_count
FROM EDIT_SIZE_DISTRIBUTION
EMIT CHANGES
LIMIT 6;

**Expected Output**:
```
+--------------+------------+
|SIZE_CATEGORY |EDIT_COUNT  |
+--------------+------------+
|Small         |80          |
|Medium        |50          |
|Large         |20          |
|Small         |90          |
|Medium        |60          |
|Large         |25          |
+--------------+------------+
```

### Exercise 11: Count Category Edits by Wiki
#### What to Do
- Create a ksqlDB table to count category edits by `wiki` in 5-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE CATEGORY_EDIT_COUNTS AS
  SELECT
    wiki,
    COUNT(*) AS category_edit_count
  FROM CATEGORY_EDITS
  WINDOW TUMBLING (SIZE 5 MINUTES)
  GROUP BY wiki
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT wiki, category_edit_count
FROM CATEGORY_EDIT_COUNTS
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+-------+-------------------+
|WIKI   |CATEGORY_EDIT_COUNT|
+-------+-------------------+
|enwiki |15                 |
|eswiki |5                  |
|frwiki |3                  |
|dewiki |7                  |
|jawiki |2                  |
+-------+-------------------+
```

### Exercise 12: Detect Edit Wars
#### What to Do
- Create a ksqlDB table to identify potential edit wars in `MAIN_NAMESPACE`.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE EDIT_WARS AS
  SELECT
    title,
    COUNT(*) AS edit_count,
    COUNT_DISTINCT(user) AS editor_count
  FROM MAIN_NAMESPACE
  WINDOW TUMBLING (SIZE 1 MINUTE)
  GROUP BY title
  HAVING COUNT(*) >= 3
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT title, edit_count, editor_count
FROM EDIT_WARS
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+------------------------+------------+-------------+
|TITLE                  |EDIT_COUNT  |EDITOR_COUNT |
+------------------------+------------+-------------+
|Climate change         |8           |5            |
|Political election     |6           |4            |
|Quantum mechanics      |5           |3            |
+------------------------+------------+-------------+
```

### Exercise 13: Detect Rapid Edits by User
#### What to Do
- Create a ksqlDB table to identify users with rapid edits in `MAIN_NAMESPACE`.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE RAPID_EDITORS AS
  SELECT
    user,
    COUNT(*) AS edit_count
  FROM MAIN_NAMESPACE
  WINDOW TUMBLING (SIZE 1 MINUTE)
  GROUP BY user
  HAVING COUNT(*) >= 3
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT user, edit_count
FROM RAPID_EDITORS
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+------------------+------------+
|USER              |EDIT_COUNT  |
+------------------+------------+
|FastEditor        |5           |
|WikiUpdater       |4           |
|TechContributor   |3           |
+------------------+------------+
```

### Exercise 14: Detect Anomalous Edits
#### What to Do
- Create a ksqlDB stream to detect anomalous edits in `MAIN_NAMESPACE`.
- Run in ksqlDB CLI:

In [ ]:
CREATE OR REPLACE STREAM ANOMALOUS_EDITS AS
  SELECT
    title,
    user,
    size_change,
    CASE
      WHEN ABS(size_change) > 500 THEN 'Huge Change'
      WHEN title IS NULL OR title = '' THEN 'Missing Title'
      WHEN user IS NULL OR user = '' THEN 'Missing User'
    END AS anomaly_type
  FROM MAIN_NAMESPACE
  WHERE ABS(size_change) > 500 OR
        title IS NULL OR title = '' OR
        user IS NULL OR user = ''
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT anomaly_type, title, user
FROM ANOMALOUS_EDITS
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+----------------+-------------------------+----------------+
| ANOMALY_TYPE   | TITLE                   | USER           |
+----------------+-------------------------+----------------+
| Huge Change    | Wp/sgh/Ewropa           | DaveZ123       |
| Huge Change    | Q136342909              | Zzhtju         |
| Huge Change    | عمر محمد الطيب           | ناعم فاروق |
+----------------+-------------------------+----------------+
```

### Exercise 15: Track Top Edited Wikis
#### What to Do
- Create a ksqlDB table to identify the top wikis by edit count in 10-minute tumbling windows.
- Run in ksqlDB CLI:

In [ ]:
CREATE TABLE TOP_WIKIS AS
  SELECT
    wiki,
    COUNT(*) AS edit_count
  FROM WIKI_RAW_STREAM
  WINDOW TUMBLING (SIZE 10 MINUTES)
  GROUP BY wiki
  EMIT CHANGES;

- Query the table:
- Now run this:

In [ ]:
SELECT wiki, edit_count
FROM TOP_WIKIS
WHERE edit_count > 100
EMIT CHANGES
LIMIT 5;

**Expected Output**:
```
+-------+------------+
|WIKI   |EDIT_COUNT  |
+-------+------------+
|enwiki |200         |
|eswiki |150         |
+-------+------------+
```

### Exercise 16: Detect Edits with Long Comments in English Wikipedia
#### What to Do
- Create a ksqlDB stream to identify edits with comments longer than 100 characters in `WIKI_RAW_STREAM` for the English Wikipedia (`enwiki`).
- Run in ksqlDB CLI:

In [ ]:
CREATE STREAM LONG_COMMENT_EDITS AS
  SELECT
    title,
    user,
    comment,
    ROWTIME AS timestamp
  FROM WIKI_RAW_STREAM
  WHERE LEN(comment) > 100 AND wiki = 'enwiki'
  EMIT CHANGES;

- Query the stream:
- Now run this:

In [ ]:
SELECT title, user, comment
FROM LONG_COMMENT_EDITS
EMIT CHANGES
LIMIT 3;

**Expected Output**:
```
+------------------------+------------------+------------------------------------------------+
|TITLE                  |USER              |COMMENT                                         |
+------------------------+------------------+------------------------------------------------+
|Climate change         |EcoEditor         |Updated with new research on global warming...  |
|Artificial intelligence|DataScientist     |Added detailed section on neural networks...    |
|Quantum mechanics      |SciEditor         |Revised quantum theory explanation with...      |
+------------------------+------------------+------------------------------------------------+
```

## Cleanup (Optional)
- Run in ksqlDB CLI to clean up objects:
- Drop streams and tables created during exercises to free resources.

In [ ]:
DROP TABLE IF EXISTS NAMESPACE_COUNTS;
DROP TABLE IF EXISTS SERVER_ACTIVITY;
DROP TABLE IF EXISTS NEW_PAGE_COUNTS;
DROP TABLE IF EXISTS FREQUENT_EDITORS;
DROP TABLE IF EXISTS EDIT_SIZE_DISTRIBUTION;
DROP TABLE IF EXISTS CATEGORY_EDIT_COUNTS;
DROP TABLE IF EXISTS EDIT_WARS;
DROP TABLE IF EXISTS RAPID_EDITORS;
DROP TABLE IF EXISTS TOP_WIKIS;
DROP STREAM IF EXISTS MAIN_NAMESPACE;
DROP STREAM IF EXISTS NEW_PAGES;
DROP STREAM IF EXISTS CATEGORY_EDITS;
DROP STREAM IF EXISTS MINOR_EDITS_LESS_THAN_100;
DROP STREAM IF EXISTS NO_COMMENT_EDITS;
DROP STREAM IF EXISTS ANOMALOUS_EDITS;
DROP STREAM IF EXISTS LONG_COMMENT_EDITS;

**Note**:
- Ensure all queries are terminated (Ctrl+C) before dropping objects.
- Save your notebook as `solutions/05_solutions.ipynb`.
- Verify services with `docker ps` and check logs if errors occur: `docker logs connect` or `docker logs ksql-server`.
- Stop containers with `docker compose down` if needed.